# Firm-Wide Economic Capital Demo

**Economic Capital Simulator** – Full Firm-Wide Aggregation  
Ajayvir Khara | Passed FRM Part I and Part II | January 2026

This notebook demonstrates the **complete firm-wide Economic Capital simulation** across:

- **Market Risk** – Multi-asset VaR/ES with Student-t shocks
- **Credit Risk** – Counterparty exposure + WWR-aware EL/UL
- **Operational Risk** – LDA with hybrid severity + expert judgment overlay

Key features:
- Full 750,000-path Monte Carlo with **t-copula** (df=3) for realistic tail dependence
- Diversification benefit calculation
- Euler (marginal) allocation of total EC to each risk type
- Automated generation of **regulatory-grade Excel report**

**Expected runtime**: ~3–6 minutes on a standard laptop

In [ ]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import sys
import os
import warnings
import glob
import time

warnings.filterwarnings("ignore", category=UserWarning)

# Styling
plt.style.use("seaborn-v0_8-pastel")
sns.set_palette("husl")
plt.rcParams["figure.figsize"] = (14, 7)
%matplotlib inline

# Project root detection
notebook_dir = Path.cwd().resolve()
if notebook_dir.name == "notebooks":
    root = notebook_dir.parent
else:
    root = notebook_dir
if str(root) not in sys.path:
    sys.path.insert(0, str(root))  # Force this into the front of the path
os.chdir(root)  # Change the working directory to the root

print(f"Current Working Directory fixed to: {os.getcwd()}")

In [ ]:
from econ_capital import run_full_simulation

## Run Full Firm-Wide Simulation

In [ ]:
%%time

from datetime import datetime

print(f"Starting firm-wide simulation at {datetime.now():%Y-%m-%d %H:%M:%S}\n")

results = run_full_simulation()

EL_total = results["EL_total"]
UL_portfolio = results["UL_portfolio"]
EC_total = results["EC_total"]
div_benefit = results["diversification_benefit"]
marginal = results["marginal_contributions"]
individual = results["individual_risks"]

## Summary Results

In [ ]:
print("=" * 70)
print("FIRM-WIDE ECONOMIC CAPITAL SUMMARY")
print("=" * 70)
print(f"{'Expected Loss (EL)':<35} : £{EL_total:>18,.0f}")
print(f"{'Portfolio Unexpected Loss (UL)':<35} : £{UL_portfolio:>18,.0f}")
print(f"{'Total Economic Capital (99.9%)':<35} : £{EC_total:>18,.0f}")
print(
    f"{'Diversification Benefit':<35} : £{div_benefit:>18,.0f} ({div_benefit / EC_total * 100:.1f}%)"
)
print("\nMarginal Contributions to Total EC:")
for risk, contrib in marginal.items():
    pct = contrib / EC_total * 100 if EC_total > 0 else 0
    print(f"   • {risk:<12}: £{contrib:>15,.0f} ({pct:>6.1f}%)")
print("=" * 70)

## Visualisation: Marginal Contribution Breakdown

In [ ]:
contrib_df = pd.DataFrame(
    {
        "Risk Type": list(marginal.keys()),
        "Marginal EC (£m)": [v / 1e6 for v in marginal.values()],
        "% of Total": [
            (v / EC_total * 100) if EC_total > 0 else 0 for v in marginal.values()
        ],
    }
).sort_values("Marginal EC (£m)", ascending=False)

plt.figure(figsize=(12, 6))
bars = sns.barplot(data=contrib_df, x="Risk Type", y="Marginal EC (£m)")
plt.title("Marginal Contribution to Firm-Wide Economic Capital")
plt.ylabel("Marginal EC (£ million)")
plt.xlabel("Risk Type")
plt.xticks(rotation=45, ha="right")
plt.grid(axis="y", alpha=0.3)

# Add percentage labels
for bar, pct in zip(bars.patches, contrib_df["% of Total"]):
    height = bar.get_height()
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        height + height * 0.02,
        f"{pct:.1f}%",
        ha="center",
        va="bottom",
    )

plt.tight_layout()
plt.show()

## Generate Regulatory-Style Firm-Wide Report

In [ ]:
%%time

from econ_capital.firmwide_reporting import generate_firmwide_ec_report

report_path = generate_firmwide_ec_report(
    aggregated_results=results, output_dir="econ_capital/reports"
)

time.sleep(1)  # Give filesystem time to update

report_pattern = str(
    root / "econ_capital" / "reports" / "firmwide" / "FirmWide_EC_Report_*.xlsx"
)
reports = glob.glob(report_pattern)

if reports:
    latest_report = max(reports, key=os.path.getctime)
    print(f"Opening latest report: {os.path.basename(latest_report)}")
    os.startfile(latest_report)  # Windows only
else:
    print("No report found in the reports directory.")

## Next Steps / Experiments

- Change `copula_df` in `generate_firmwide_ec_report()` (try 2.0 for fatter tails, 10+ for near-Gaussian)
- Modify correlations in `default.yaml` to observe diversification benefit changes
- Toggle WWR on/off in Credit Risk config and re-run
- Increase simulation paths to 1M+ for ultra-stable marginals
- Run sensitivity to confidence level (97.5% → 99.97%) by modifying `aggregate_economic_capital()` calls

See the other notebooks:
- `demo_credit.ipynb`
- `demo_market.ipynb`
- `demo_oprisk.ipynb`